In [ ]:
#clone SSD GitHub repo
!git clone https://github.com/weiliu89/caffe.git

In [ ]:
#import dependencies and functions
!pip install seaborn
import tensorflow as tf
from tensorflow.python.ops import control_flow_op
from datasets import dataset_factory
from deployment import model_deploy
from nets import nets_factory
from preprocessing import preprocessing_factory
import tf_utils
import os
import math
import random
import numpy as np
import tensorflow as tf
import cv2
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from nets import ssd_vgg_300, ssd_common, np_methods
from preprocessing import ssd_vgg_preprocessing
from notebooks import visualization

slim = tf.contrib.slim

In [ ]:
#LOAD DATA

%cd /SSD/caffe

slim = tf.contrib.slim

DATA_FORMAT = 'NCHW'

DATASET_DIR='/Applications/anaconda3/envs/COMP6970/FinalProject/SSD/DataSSD'
TRAIN_DIR='/Applications/anaconda3/envs/COMP6970/FinalProject/SSD/DataSSD/train_20shot'
CHECKPOINT_PATH='/Applications/anaconda3/envs/COMP6970/FinalProject/SSD/checkpoints/ssd_300_vgg.ckpt'

In [ ]:
#TRAIN MODEL

python train_ssd_network.py --train_dir=${TRAIN_DIR} --dataset_dir=${DATASET_DIR} --dataset_name=algae --dataset_split_name=train 
--model_name=ssd_300_vgg --checkpoint_path=${CHECKPOINT_PATH} --save_summaries_secs=60 --save_interval_secs=600 --weight_decay=0.0005 
--optimizer=adam --learning_rate=0.001 --batch_size=32 --epochs=10 --gpu_memory_fraction=0.9 

In [ ]:
#TEST MODEL

# Input placeholder.
net_shape = (300, 300)
data_format = 'NHWC'
img_input = tf.placeholder(tf.uint8, shape=(None, None, 3))

# Evaluation pre-processing: resize to SSD net shape.
image_pre, labels_pre, bboxes_pre, bbox_img = ssd_vgg_preprocessing.preprocess_for_eval(
    img_input, None, None, net_shape, data_format, resize=ssd_vgg_preprocessing.Resize.WARP_RESIZE)
image_4d = tf.expand_dims(image_pre, 0)

# Define the SSD model.
reuse = True if 'ssd_net' in locals() else None
ssd_net = ssd_vgg_300.SSDNet()
with slim.arg_scope(ssd_net.arg_scope(data_format=data_format)):
    predictions, localisations, _, _ = ssd_net.net(image_4d, is_training=False, reuse=reuse)

# Use the SSD model trained on algae data
ckpt_filename = '/Applications/anaconda3/envs/COMP6970/FinalProject/SSD/log/model.ckpt-222914' #change directory as needed

isess.run(tf.global_variables_initializer())

# SSD default anchor boxes.
ssd_anchors = ssd_net.anchors(net_shape)

# Function to process images with SSD
def process_image(img, select_threshold=0.502, nms_threshold=.45, net_shape=(300, 300)):
    # Run SSD network.
    rimg, rpredictions, rlocalisations, rbbox_img = isess.run([image_4d, predictions, localisations, bbox_img],
                                                              feed_dict={img_input: img})
    
    # Get classes and bboxes from the net outputs.
    rclasses, rscores, rbboxes = np_methods.ssd_bboxes_select(
            rpredictions, rlocalisations, ssd_anchors,
            select_threshold=select_threshold, img_shape=net_shape, num_classes=21, decode=True)
    
    rbboxes = np_methods.bboxes_clip(rbbox_img, rbboxes)
    rclasses, rscores, rbboxes = np_methods.bboxes_sort(rclasses, rscores, rbboxes, top_k=400)
    rclasses, rscores, rbboxes = np_methods.bboxes_nms(rclasses, rscores, rbboxes, nms_threshold=nms_threshold)
    

    rbboxes = np_methods.bboxes_resize(rbbox_img, rbboxes)
    return rclasses, rscores, rbboxes

#test image batches; outputs class, bounding box, scores, and metrics
# run on test images and visualize output.
path = '/Applications/anaconda3/envs/COMP6970/FinalProject/SSD/DataSSD/test/'
image_names = sorted(os.listdir(path))

img = cv2.imread(path + image_names[-2])

rclasses, rscores, rbboxes =  process_image(img) #outputs results of testing

visualization.plt_bboxes(img, rclasses, rscores, rbboxes)
